# Cattle Breed Recognition AI 🐄

This notebook represents the final, High-Accuracy Image Processing academic pipeline. The dataset inside Google Drive is already safely pre-split into `train`, `val`, and `test` folders.

## 1. Cloud Storage Connection
Unzip the pre-organized dataset directly into Colab's fast memory.

In [ ]:
# !unzip -q "/content/drive/MyDrive/ImageProcessing_Project/data.zip" -d "/content/dataset"
# from google.colab import drive
# drive.mount('/content/drive')

## 2. Imports and Configuration
We load TensorFlow and set the training runway for a massive 30 Epochs.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Constants
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
TRAIN_DIR = '/content/dataset/data/train'
VAL_DIR = '/content/dataset/data/val'
TEST_DIR = '/content/dataset/data/test'
class_names = ['Gir', 'Red_Sindhi', 'Sahiwal', 'Tharparkar']
EPOCHS = 30

## 3. Data Loading
TensorFlow locks onto the specific directories for Train, Validation, and Testing.

In [ ]:
print("Loading training data...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

print("Loading validation data...")
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    shuffle=False, # Essential for plotting matrix later
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

print("Loading testing data...")
test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    shuffle=False, # Essential for plotting matrix later
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

## 4. Advanced Data Augmentation
Fixes low-accuracy caused by bad lighting (Contrast) and off-center placement (Translation/Zoom).

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1)
])

def format_image_dataset(imgs, labels):
    return preprocess_input(imgs), labels

train_dataset = train_dataset.map(lambda x, y: (data_augmentation(x, training=True), y))
train_dataset = train_dataset.map(format_image_dataset, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

validation_dataset = validation_dataset.map(format_image_dataset, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.map(format_image_dataset, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

## 5. Fine-Tuning MobileNetV2 Architecture
Instead of freezing the entire network, we unlock the top 24 layers to let the AI learn the specific difference between cattle wrinkles and human background noise.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Unlocking the brain!
base_model.trainable = True
fine_tune_at = 130
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x) # Strong 128-node intermediate brain
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

## 6. Model Training (30 Epochs)
This implements a Learning Rate Scheduler to automatically "downshift" if the accuracy gets stuck.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1
)

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=validation_dataset,
    callbacks=[lr_scheduler]
)

model.save('cattle_breed_mobilenetv2.h5')
print("Model Successfully Saved -> cattle_breed_mobilenetv2.h5")

## 7. Results & Graphs
Visually confirming that we did not overfit.

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
print("Generating predictions on pure TEST Dataset...")
test_images = []
test_labels = []

for batch_imgs, batch_labels in test_dataset:
    test_images.extend(batch_imgs.numpy())
    test_labels.extend(batch_labels.numpy())

test_images = np.array(test_images)
test_labels = np.array(test_labels)

test_preds = model.predict(test_images, batch_size=BATCH_SIZE)
test_preds_classes = np.argmax(test_preds, axis=1)

cm = confusion_matrix(test_labels, test_preds_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('TEST Confusion Matrix')
plt.show()

print("\nTEST Classification Report:")
print(classification_report(test_labels, test_preds_classes, target_names=class_names))

## 8. Real-Time Inference Visualizer
Automatically grabs a random test picture, displays it beautifully, and prints the model's actual real-life prediction against the Truth. Great for academic demonstrations.

In [ ]:
def predict_image(img_path, model_path='cattle_breed_mobilenetv2.h5'):
    loaded_model = tf.keras.models.load_model(model_path)
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    predictions = loaded_model.predict(img_array)
    predicted_class = class_names[np.argmax(predictions)]
    confidence = np.max(predictions)
    return predicted_class.lower()

# 1. Grab a random test image from exactly the pure test directory
true_breed = random.choice(class_names)
breed_folder = os.path.join(TEST_DIR, true_breed)
valid_images = [f for f in os.listdir(breed_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.JPG'))]
random_image_file = random.choice(valid_images)
sample_image_path = os.path.join(breed_folder, random_image_file)

# 2. Display the Image
img_to_show = image.load_img(sample_image_path)
plt.figure(figsize=(4, 4))
plt.imshow(img_to_show)
plt.axis('off')
plt.title(f"Actual Truth: {true_breed.upper()}")
plt.show()

# 3. Run AI Inference
print("Running AI Inference...\n")
predicted_breed = predict_image(sample_image_path)

# 4. Check status
if predicted_breed == true_breed.lower():
    print(f"\n✅ SUCCESS: The model correctly identified this as a {predicted_breed.upper()}!")
else:
    print(f"\n❌ FAILED: The model guessed {predicted_breed.upper()} but it was actually {true_breed.upper()}.")